# Week 6 — Advanced Retrieval: Transform and Measure

This notebook extends the Week 5 retrieval pipeline using the same 16 chunks and the same 10 frozen evaluation queries.

It follows the assignment flow:

1. Recreate the Week 5 baseline retriever.
2. Apply a Gemini multi-query transformation.
3. Measure transformation-only retrieval.
4. Re-rank transformed candidates with a local cross-encoder.
5. Compare baseline, transformed, and final results per query.
6. Identify which query types benefited and which did not.
7. Find a transformation regression, or justify a no-regression result.

The optional schema-constrained extraction enrichment is not included.


## 1. Setup

The notebook is designed for Google Colab or local JupyterLab on CPU.

Credential lookup order:

1. Colab Secrets: `GEMINI_API_KEY`
2. Existing environment variable
3. Local `.env` file

The Gemini transformation returns **two rewrites in one call per query**, so an empty cache requires at most 10 live Gemini requests. Responses are cached in `week6_query_cache.json`.


In [ ]:
import sys, subprocess, importlib.util

mods = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "google.genai": "google-genai",
    "dotenv": "python-dotenv",
}

need = [package for module, package in mods.items() if importlib.util.find_spec(module) is None]
if need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *need])


In [ ]:
import os
import json
import time
import math
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import faiss

from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display, Markdown

TOP_K = 3
CANDIDATE_K = 8

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEMINI_MODEL = "gemini-3.5-flash-lite"

CACHE_PATH = Path("week6_query_cache.json")


## 2. Week 5 corpus and frozen evaluation set

The corpus and gold labels are kept unchanged from Week 5 so the before/after comparison is valid.


In [ ]:
DOCUMENTS = [
    ("D01", "Request intake", [
        ("S1", "Required fields",
         "A new operations request must include a short title, requester name, business team, problem statement, and desired outcome. The requester can attach example files but attachments are optional."),
        ("S2", "Duplicate requests",
         "Before opening a new request, search the request list by title and team. If an existing request covers the same problem, add a comment to that request rather than creating a duplicate."),
    ]),
    ("D02", "Triage and priority", [
        ("S1", "Priority definitions",
         "Priority 1 means a production service is unavailable to multiple people. Priority 2 means a blocked business process with a manual workaround. Priority 3 means an improvement or a question without an active outage."),
        ("S2", "First review",
         "The operations coordinator checks completeness and assigns an owner within two business days. Missing intake fields are sent back to the requester before priority is assigned."),
    ]),
    ("D03", "Approval workflow", [
        ("S1", "Approval conditions",
         "A request that changes customer-facing content or grants access to protected data requires manager approval before implementation. An internal wording correction with no data access change does not require approval."),
        ("S2", "Rejected approval",
         "When a manager rejects a request, record the reason and set approval result to Rejected. The owner may revise the request and seek a new approval; rejection does not mean it is complete."),
    ]),
    ("D04", "Delivery and review", [
        ("S1", "Human review",
         "Generated drafts must be reviewed for factual correctness by the assigned owner. A workflow must not automatically send a draft to customers or publish it internally before a person approves the final text."),
        ("S2", "Completion",
         "Set status to Completed only after the requester accepts the delivered result and the owner records the outcome. Saving a generated draft alone does not close a request."),
    ]),
    ("D05", "SharePoint file handling", [
        ("S1", "Source files",
         "Store source files in the team SharePoint document library. In the request, link the specific file and record its version; do not paste the full file into the request description."),
        ("S2", "Access permissions",
         "Requesters may see their own request and shared team files. An owner checks a recipient's access before sharing a source link; a link does not grant permission by itself."),
    ]),
    ("D06", "Automation monitoring", [
        ("S1", "Flow failure",
         "If a Power Automate run fails, inspect its run history and note the failed action and error in the request. Retry only after correcting the cause, to avoid sending duplicate notifications."),
        ("S2", "Success checks",
         "For a new automation, test one ordinary case and one missing-field case. Record expected versus observed results and ask the owner to sign off before activation."),
    ]),
    ("D07", "Data safeguards", [
        ("S1", "Sensitive content",
         "Remove customer identifiers and confidential attachments from example prompts unless their use is authorized. Use a small, sanitized excerpt when a full document is unnecessary."),
        ("S2", "Retention",
         "Keep the request's decision and final outcome in the request list. Delete temporary test exports when validation finishes according to the team's retention schedule."),
    ]),
    ("D08", "Notifications and escalation", [
        ("S1", "Status updates",
         "The owner posts a status update on the request when work starts and when the expected delivery date changes. Routine updates are visible to the requester in the list."),
        ("S2", "Escalation",
         "Escalate a production outage to the on-call owner immediately and record the escalation time. A non-outage item waiting more than two business days for assignment goes to the operations coordinator."),
    ]),
]

CHUNKS = [
    {
        "chunk_id": f"{doc_id}-{section_id}",
        "title": title,
        "heading": heading,
        "body": body,
        "text": f"{title} — {heading}. {body}",
    }
    for doc_id, title, sections in DOCUMENTS
    for section_id, heading, body in sections
]

by_id = {chunk["chunk_id"]: chunk for chunk in CHUNKS}

TESTS = [
    ("Q01", "What fields must a requester provide when opening a request?", {"D01-S1"}),
    ("Q02", "What should I do if someone has already reported the same problem?", {"D01-S2"}),
    ("Q03", "Which priority applies when a process is blocked but a workaround exists?", {"D02-S1"}),
    ("Q04", "How soon is an owner assigned, and what happens to an incomplete request?", {"D02-S2"}),
    ("Q05", "When does a request require manager approval?", {"D03-S1"}),
    ("Q06", "What happens after a manager rejects a request?", {"D03-S2"}),
    ("Q07", "Can an AI draft be published or emailed without a person checking it?", {"D04-S1"}),
    ("Q08", "When can the owner mark the request Completed?", {"D04-S2"}),
    ("Q09", "How do I investigate a failed Power Automate flow before retrying?", {"D06-S1"}),
    ("Q10", "How quickly should a production outage be escalated?", {"D08-S2"}),
]

display(pd.DataFrame([
    {"qid": qid, "query": query, "gold": next(iter(gold))}
    for qid, query, gold in TESTS
]))


## 3. Week 5 baseline retriever

MiniLM is used as the baseline because its saved Week 5 run ranked the gold chunk first on 8/10 queries, while MPNet was already 10/10. Using MiniLM leaves room to measure ranking improvement instead of starting at a ceiling.


In [ ]:
embedder = SentenceTransformer(EMBED_MODEL)

texts = [chunk["text"] for chunk in CHUNKS]
vectors = embedder.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=False
).astype("float32")

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

def retrieve(query, k=TOP_K):
    qv = embedder.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")
    scores, ids = index.search(qv, k)
    return [
        {
            "chunk_id": CHUNKS[j]["chunk_id"],
            "score": float(score),
            "rank": rank + 1,
        }
        for rank, (j, score) in enumerate(zip(ids[0], scores[0]))
    ]

baseline = {
    qid: retrieve(query, TOP_K)
    for qid, query, _ in TESTS
}

display(pd.DataFrame([
    {
        "qid": qid,
        "baseline_ids": [x["chunk_id"] for x in baseline[qid]],
    }
    for qid, _, _ in TESTS
]))


## 4. Query transformation

The transformation is **multi-query rewriting**. Gemini produces two alternate search queries that preserve the original intent but use likely operations-policy vocabulary.

The original query is kept alongside both rewrites during fusion. This reduces the risk that one imperfect rewrite completely replaces the user's wording.


In [ ]:
def load_gemini_key():
    # 1. Colab Secrets first.
    try:
        from google.colab import userdata
        for name in ("GEMINI_API_KEY", "GEMENI_API_KEY"):
            try:
                value = userdata.get(name)
                if value:
                    return value.strip(), f"Colab Secrets ({name})"
            except Exception:
                pass
    except ImportError:
        pass

    # 2. Existing environment variable.
    for name in ("GEMINI_API_KEY", "GEMENI_API_KEY"):
        value = os.getenv(name)
        if value:
            return value.strip(), f"environment variable ({name})"

    # 3. Local JupyterLab .env.
    env_file = find_dotenv(usecwd=True)
    if env_file:
        load_dotenv(env_file, override=False)
        for name in ("GEMINI_API_KEY", "GEMENI_API_KEY"):
            value = os.getenv(name)
            if value:
                return value.strip(), f".env ({Path(env_file).name}, {name})"

    return None, None

cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
key, key_source = load_gemini_key()

needs_live_calls = any(
    len(cache.get(qid, [])) < 2
    for qid, _, _ in TESTS
)

if key_source:
    print(f"Gemini API key loaded from {key_source}.")
elif needs_live_calls:
    raise RuntimeError(
        "Gemini API key not found. Add GEMINI_API_KEY to Colab Secrets "
        "(with notebook access enabled), or set GEMINI_API_KEY in a local .env file."
    )
else:
    print("Complete query cache found; Gemini API key is not required for this run.")


In [ ]:
if key and needs_live_calls:
    from google import genai

    client = genai.Client(api_key=key)

    def generate_rewrites(query, max_retries=4):
        prompt = f'''
Rewrite the semantic-search query below two different ways using likely
operations-policy vocabulary.

Requirements:
- Preserve the original intent exactly.
- Produce exactly two concise search queries.
- Do not explain them.
- Return one query per line.
- Prefix them with 1. and 2.

Original query:
{query}
'''.strip()

        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=prompt,
                )

                lines = [
                    re.sub(r"^\s*\d+[\.\):\-]\s*", "", line).strip()
                    for line in response.text.splitlines()
                    if line.strip()
                ]
                rewrites = [line.strip('\"') for line in lines if line]

                if len(rewrites) >= 2:
                    return rewrites[:2]

                raise ValueError(
                    f"Gemini returned fewer than two rewrites: {response.text!r}"
                )

            except Exception as exc:
                message = str(exc)
                if "429" in message or "RESOURCE_EXHAUSTED" in message:
                    match = re.search(
                        r"retry in ([0-9.]+)s",
                        message,
                        flags=re.IGNORECASE,
                    )
                    delay = float(match.group(1)) + 2 if match else 40
                    print(f"Rate limit reached. Retrying in {delay:.0f} seconds...")
                    time.sleep(delay)
                    continue
                raise

        raise RuntimeError(
            "Gemini rate limit persisted after automatic retries."
        )

    for qid, query, _ in TESTS:
        existing = cache.get(qid, [])
        if len(existing) >= 2:
            continue

        print(f"Generating rewrites for {qid}...")
        generated = generate_rewrites(query)

        combined = list(existing)
        for rewrite in generated:
            if rewrite not in combined:
                combined.append(rewrite)
            if len(combined) == 2:
                break

        cache[qid] = combined[:2]
        CACHE_PATH.write_text(json.dumps(cache, indent=2))

        # Keeps a clean 10-query run comfortably below the observed free-tier RPM limit.
        time.sleep(2)

transformed = {
    qid: cache[qid][:2]
    for qid, _, _ in TESTS
}

display(pd.DataFrame([
    {
        "qid": qid,
        "original": query,
        "rewrite_1": transformed[qid][0],
        "rewrite_2": transformed[qid][1],
    }
    for qid, query, _ in TESTS
]))


## 5. Transformation-only retrieval

This stage measures the query transformation **before** cross-encoder re-ranking.

The original query and its two rewrites each retrieve candidates. Reciprocal Rank Fusion (RRF) combines those ranked lists. The top three fused chunks are the transformation-only result.

Keeping this stage separate is important: otherwise a strong reranker could repair a poor transformation and hide the transformation failure the assignment asks us to inspect.


In [ ]:
def rrf(rank_lists, k=60):
    scores = defaultdict(float)
    for rank_list in rank_lists:
        for item in rank_list:
            scores[item["chunk_id"]] += 1 / (k + item["rank"])

    return [
        chunk_id
        for chunk_id, _ in sorted(
            scores.items(),
            key=lambda item: item[1],
            reverse=True,
        )
    ]

def transform_candidates(query, rewrites):
    rank_lists = [
        retrieve(query, CANDIDATE_K),
        *[retrieve(rewrite, CANDIDATE_K) for rewrite in rewrites],
    ]
    return rrf(rank_lists)[:CANDIDATE_K]

transformed_candidates = {
    qid: transform_candidates(query, transformed[qid])
    for qid, query, _ in TESTS
}

transform_only = {
    qid: candidates[:TOP_K]
    for qid, candidates in transformed_candidates.items()
}

display(pd.DataFrame([
    {
        "qid": qid,
        "transform_only_ids": transform_only[qid],
    }
    for qid, _, _ in TESTS
]))


## 6. Cross-encoder re-ranking

The local cross-encoder scores each fused candidate against the **original user query**, then sorts candidates by that relevance score. This adds latency because each query-candidate pair is evaluated jointly, but it can improve ordering when the bi-encoder retrieves several semantically related chunks.


In [ ]:
reranker = CrossEncoder(RERANK_MODEL)

def rerank(query, candidate_ids):
    pairs = [
        (query, by_id[chunk_id]["text"])
        for chunk_id in candidate_ids
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidate_ids, scores),
        key=lambda item: float(item[1]),
        reverse=True,
    )

    return [chunk_id for chunk_id, _ in ranked[:TOP_K]]

finals = {
    qid: rerank(query, transformed_candidates[qid])
    for qid, query, _ in TESTS
}

display(pd.DataFrame([
    {
        "qid": qid,
        "final_ids": finals[qid],
    }
    for qid, _, _ in TESTS
]))


## 7. Before-and-after measurement

Metrics:

- **Precision@3** — fraction of the top 3 that are relevant.
- **Recall@3** — whether the relevant chunk appears in the top 3.
- **Hit@1** — whether the gold chunk is ranked first.
- **MRR** — reciprocal rank of the first relevant result.
- **nDCG@3** — ranking-sensitive gain within the top 3.

The table reports all three stages per query:

1. Week 5 baseline
2. Transformation-only RRF
3. Transformation + cross-encoder reranking


In [ ]:
def metrics(ids, gold):
    hits = [int(chunk_id in gold) for chunk_id in ids[:TOP_K]]

    reciprocal_rank = next(
        (
            1 / rank
            for rank, chunk_id in enumerate(ids[:TOP_K], start=1)
            if chunk_id in gold
        ),
        0,
    )

    dcg = sum(
        rel / math.log2(rank + 1)
        for rank, rel in enumerate(hits, start=1)
    )

    ideal_hits = sorted(hits, reverse=True)
    idcg = sum(
        rel / math.log2(rank + 1)
        for rank, rel in enumerate(ideal_hits, start=1)
    )

    ndcg = dcg / idcg if idcg else 0

    return {
        "p3": sum(hits) / TOP_K,
        "r3": sum(hits) / len(gold),
        "hit1": hits[0] if hits else 0,
        "mrr": reciprocal_rank,
        "ndcg3": ndcg,
    }

rows = []

for qid, query, gold in TESTS:
    baseline_ids = [item["chunk_id"] for item in baseline[qid]]
    transformed_ids = transform_only[qid]
    final_ids = finals[qid]

    base_m = metrics(baseline_ids, gold)
    transform_m = metrics(transformed_ids, gold)
    final_m = metrics(final_ids, gold)

    row = {
        "qid": qid,
        "query": query,
        "gold": next(iter(gold)),
        "baseline_ids": baseline_ids,
        "transform_ids": transformed_ids,
        "final_ids": final_ids,
    }

    for name, value in base_m.items():
        row[f"base_{name}"] = value

    for name, value in transform_m.items():
        row[f"transform_{name}"] = value
        row[f"delta_transform_{name}"] = value - base_m[name]

    for name, value in final_m.items():
        row[f"final_{name}"] = value
        row[f"delta_final_{name}"] = value - base_m[name]
        row[f"delta_rerank_{name}"] = value - transform_m[name]

    rows.append(row)

results = pd.DataFrame(rows)

display(results[[
    "qid",
    "query",
    "gold",
    "baseline_ids",
    "transform_ids",
    "final_ids",
    "base_hit1",
    "transform_hit1",
    "final_hit1",
    "delta_transform_mrr",
    "delta_final_mrr",
]])

metric_names = ["p3", "r3", "hit1", "mrr", "ndcg3"]

summary = pd.DataFrame(
    {
        "baseline": [results[f"base_{m}"].mean() for m in metric_names],
        "transform_only": [results[f"transform_{m}"].mean() for m in metric_names],
        "final_reranked": [results[f"final_{m}"].mean() for m in metric_names],
    },
    index=metric_names,
)

summary["transform_delta"] = summary["transform_only"] - summary["baseline"]
summary["final_delta"] = summary["final_reranked"] - summary["baseline"]
summary["rerank_delta"] = summary["final_reranked"] - summary["transform_only"]

display(summary.round(3))


## 8. Which query types benefited?

Rather than infer categories from the metric alone, this section groups the observed cases by their baseline behavior.

- **Baseline-miss at rank 1:** the gold chunk was present in the top 3 but not ranked first. These queries have the clearest opportunity to benefit from better ranking.
- **Already-correct at rank 1:** the baseline already placed the gold chunk first, so improvement is limited by a ceiling.


In [ ]:
analysis_rows = []

for _, row in results.iterrows():
    if row["base_hit1"] == 0:
        query_type = "baseline gold not rank 1"
    else:
        query_type = "baseline already rank 1"

    benefited_transform = row["delta_transform_mrr"] > 0
    benefited_final = row["delta_final_mrr"] > 0

    analysis_rows.append({
        "qid": row["qid"],
        "query_type": query_type,
        "transform_mrr_delta": row["delta_transform_mrr"],
        "final_mrr_delta": row["delta_final_mrr"],
        "benefited_transform": benefited_transform,
        "benefited_final": benefited_final,
    })

query_type_results = pd.DataFrame(analysis_rows)
display(query_type_results)

grouped = query_type_results.groupby("query_type").agg(
    queries=("qid", "count"),
    mean_transform_mrr_delta=("transform_mrr_delta", "mean"),
    mean_final_mrr_delta=("final_mrr_delta", "mean"),
    transform_benefits=("benefited_transform", "sum"),
    final_benefits=("benefited_final", "sum"),
)

display(grouped.round(3))


In [ ]:
baseline_misses = query_type_results[
    query_type_results["query_type"] == "baseline gold not rank 1"
]

already_correct = query_type_results[
    query_type_results["query_type"] == "baseline already rank 1"
]

benefited_ids = query_type_results.loc[
    query_type_results["final_mrr_delta"] > 0,
    "qid"
].tolist()

unchanged_ids = query_type_results.loc[
    query_type_results["final_mrr_delta"] == 0,
    "qid"
].tolist()

display(Markdown(
    "### Query-type interpretation\n"
    f"- Baseline rank-1 misses: **{baseline_misses['qid'].tolist()}**. "
    f"These were the queries with room for ranking improvement.\n"
    f"- Queries improved by the final pipeline: **{benefited_ids if benefited_ids else 'none'}**.\n"
    f"- Queries unchanged by the final pipeline: **{unchanged_ids if unchanged_ids else 'none'}**.\n"
    "- Queries already correct at rank 1 had little or no room to improve, so unchanged "
    "MRR on those cases is expected rather than evidence that the method failed."
))


## 9. Required failure case: transformation regression

The assignment asks specifically whether the **query transformation** made retrieval worse.

Therefore this section compares the baseline directly with the **transformation-only** stage, not only with the final reranked result.


In [ ]:
transform_regressions = results[
    results["delta_transform_mrr"] < 0
].sort_values("delta_transform_mrr")

if len(transform_regressions):
    worst = transform_regressions.iloc[0]
    qid = worst["qid"]

    display(Markdown(
        f"### Transformation regression: {qid}\n"
        f"**Original query:** {worst['query']}\n\n"
        f"**Rewrites:** {transformed[qid]}\n\n"
        f"**Gold chunk:** {worst['gold']} — {by_id[worst['gold']]['text']}\n\n"
        f"**Baseline ranking:** {worst['baseline_ids']}\n\n"
        f"**Transformation-only ranking:** {worst['transform_ids']}\n\n"
        f"**Final reranked ranking:** {worst['final_ids']}\n\n"
        "**Explanation:** The transformed wording appears to have emphasized a related "
        "operations concept strongly enough to move a less-direct chunk above the gold chunk. "
        "That explanation is an inference from the changed ranking, not proof of the embedding "
        "model's internal cause. The final reranker is reported separately so that a repaired "
        "ranking does not hide the transformation regression."
    ))
else:
    display(Markdown(
        "### No transformation regression observed\n"
        "No query had a lower MRR after transformation-only RRF than under the Week 5 baseline. "
        "Across this ten-query set, the rewrites preserved the core action or entity being asked "
        "about, and the original user query remained in the fusion step alongside both rewrites. "
        "That design reduced the chance that one imperfect rewrite could displace the gold chunk. "
        "The candidate pool also remained broad (`CANDIDATE_K=8`), so related wording could add "
        "recall without forcing the original query out of consideration. This is a result for this "
        "test set only and does not prove multi-query transformation is universally safe."
    ))


## 10. Final interpretation

This cell reports the measured aggregate comparison and the latency tradeoff.


In [ ]:
b = summary["baseline"]
t = summary["transform_only"]
f = summary["final_reranked"]

display(Markdown(
    "### Measured interpretation\n"
    f"- **Baseline:** P@3 {b['p3']:.3f}, R@3 {b['r3']:.3f}, "
    f"Hit@1 {b['hit1']:.3f}, MRR {b['mrr']:.3f}, nDCG@3 {b['ndcg3']:.3f}.\n"
    f"- **Transformation-only:** P@3 {t['p3']:.3f}, R@3 {t['r3']:.3f}, "
    f"Hit@1 {t['hit1']:.3f}, MRR {t['mrr']:.3f}, nDCG@3 {t['ndcg3']:.3f}.\n"
    f"- **After cross-encoder reranking:** P@3 {f['p3']:.3f}, R@3 {f['r3']:.3f}, "
    f"Hit@1 {f['hit1']:.3f}, MRR {f['mrr']:.3f}, nDCG@3 {f['ndcg3']:.3f}.\n\n"
    f"Transformation MRR delta vs. baseline: **{summary.loc['mrr', 'transform_delta']:+.3f}**. "
    f"Final MRR delta vs. baseline: **{summary.loc['mrr', 'final_delta']:+.3f}**.\n\n"
    "The tradeoff is additional latency: an uncached run uses at most one Gemini call per query "
    "plus extra local retrieval and cross-encoder scoring. The cache removes repeated Gemini calls "
    "on later runs."
))


## 11. AI use

ChatGPT was used to debug the Week 6 notebook, preserve the Week 5 evaluation set, and grammatically check/improve documentation.

Gemini Flash-Lite is used inside the notebook for the measured query transformations. The saved measurements, transformed rankings, reranked results, and conclusions should come from executing this notebook and reviewing its outputs.
